# A.I. Lab: Computer Vision and NLP - Deep Learning

An artificial neural network is a learning model whose structure is inspired by the human brain. <br>
Nowadays, most neural networks follow the multilayer perceptron model, which provides data to the input layer, which then sends them through the hidden layer(s) for the computational steps so that, in the end, the output layer can return a classification output through a dedicated array of probabilities. <br>
Generally speaking, a multilayer perceptron is trained by repeating feedforwarding, which consists in computing the mean square error of the classification predictions, and backpropagation, which instead applies stochastic gradient descent in order to minimise future errors, until convergence.

In [ ]:
# A simple, but not efficient, Scikit-Learn implementation of a multilayer perceptron.
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Start by loading the MNIST dataset of handwritten digits and split it into training and testing datasets.
X, y = fetch_openml('mnist_784', return_X_y=True) # load the dataset array and the labels
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7) # 70% training set, 30% testing set

# At this point, create an instance of the multilayer perceptron classifier.
model = MLPClassifier(hidden_layer_sizes=(10,)) # create a MLP with one hidden layer composed by 10 neurons

# Fit the model using the training dataset.
model.fit(X_train, y_train)

# Lastly, use the testing dataset to make predictions and compute the accuracy score.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_pred=y_pred, y_true=y_test)
print(accuracy)

# N.B.: It is also possible to scale the dataset in order to achieve better results.

In [ ]:
# A harder, but more efficient, Pytorch implementation of a multilayer perceptron based on tensors.
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn
import torchmetrics

device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility

# Start by importing and downloading the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor())

# Now, define the structure of the multilayer perceptron model through neural network classes.
# In this case, the model is implemented sequentially using linear layers that are connected by a sigmoid activation function.
class MyMLP(nn.Module):
    def __init__(self):
        super().__init__() # initialize the attributes of the parent class
        self.mlp = nn.Sequential(
            nn.Linear(28*28,5),
            nn.Sigmoid(),
            nn.Linear(5,5),
            nn.Sigmoid(),
            nn.Linear(5,5),
            nn.Sigmoid(),
            nn.Linear(5,10)
        )
        self.flatten = nn.Flatten() # flatten the structure into a one-dimensional array
    def forward(self, x): # specify how the data goes inside the model by connecting the layers
        # This function specifies how the model treats the data and how the probability tensore for classification is computed.
        x = self.flatten(x) # convert the input image into a one-dimensional array
        logits = self.mlp(x) # perform a raw classification on the prepared data that will be normalized using the loss function
        return logits

# Having defined the neural network, create an instance of the MLP model.
model = MyMLP()

# Set the hyperparameters of the model.
epochs = 2 # check the whole dataset twice
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model by using cross entropy loss.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

# Define the optimizer for gradient descent.
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # perform stochastic gradient descent on the parameters

# Define the training loop for the training stage.
def training_loop(dataloader, model, loss_function, optimizer):
    dataset_size = len(dataloader)
    # Start by loading the data batch from the disk.
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X) # get the predictions of the model on the training dataset
        loss = loss_function(pred, y) # compute the prediction error using the loss function
        # Perform backpropagation in order to minimize future losses.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss:{loss}, [{current} /{dataset_size}]")
            acc = metric(pred, y)
            print(f"Accuracy of the current batch: {acc}")
    # Print the final training accuracy.
    acc = metric.compute()
    print(f"Final training accuracy: {acc}")
    metric.reset() # reset for future loops

# Define the testing loop for the testing stage (remember to disable the weights update).
def testing_loop(dataloader, loss_function, model):
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X) # get the predictions of the model on the testing dataset
            acc = metric(pred, y)
    acc = metric.compute()
    print(f"Final testing accuracy: {acc}")
    metric.reset() # reset for future loops

# Perform the actual model training and testing.
for e in range(epochs):
    print(f"Epoch: {e}")
    training_loop(train_dataloader, model, loss_function, optimizer)
    testing_loop(test_dataloader, loss_function, model)
print("Done")

**N.B.:** It is also possible to improve the performance of a neural network model by using different activation functions and/or           by using more flexible/non-sequential structures.

A convolutional neural network deals with learning image-like data through convolution operations. <br>
The general structure of a convolutional neural network consists of a convolution layer, which performs convolution on the input data in order to provide a feature map, and a pooling layer, which instead uses the feature map to "downsample" the data through a dedicated kernel, along with a fully connected layer that implements a multilayer perceptron to perform classification on the flattened feature map.

Similarly to machine learning models, a convolutional neural network requires setting some hyperparameters for the learning procedure, such as filter size, stride and padding.

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn
import torchmetrics

device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility

# Start by importing and downloading the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor())

# Define the structure of the convolutional neural network model through neural network classes.
# In particular, start by creating a sequential 2D convolutional neural network for the convolution and pooling layers and then
# add a sequential multilayer perceptron for the fully connected layer.
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=5, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(in_channels=5, out_channels=10, kernel_size=3),
            nn.ReLU()
        )
        self.flatten = nn.Flatten() # to flatten the feature map for the fully connected layer
        self.mlp = nn.Sequential(
            nn.Linear(5760, 10), # fix the input layer's size by raising an error on a random tensor
            nn.ReLU(),
            nn.Linear(10, 10)
        )
    def forward(self, x):
        x = self.cnn(x) # create a 2D feature part using the convolutional neural network
        x = self.flatten(x) # flatten the feature map so that it can be used by the multilayer perceptron
        logits = self.mlp(x) # use the MLP for classification by exploiting the features extracted from the CNN
        return logits

# At this point, create an instance of the model.
model = MyCNN()
# Since the size of the input layer of the fully connected layer may not be known a priori, it is generally convenient to derive it
# by raising an error on a randomly defined tensor.
'''
fake_input = torch.rand((1, 1, 28, 28))
model(fake_input)
'''

# Start by defining the hyperparameters of the model.
epochs = 2 # check the whole dataset twice
batch_size = 64 # check 64 data samples at once
learning_rate = 0.0001 # use a smaller learning rate in order to obtain more accurate results during the backpropagation phase

# Define the loss function for the classification model.
loss_function = nn.CrossEntropyLoss() # choose cross entropy loss

# Use a dataloader for processing the data in batches.
train_dataloader = DataLoader(training_data, batch_size=batch_size) # dataloader for the training dataset
test_dataloader = DataLoader(test_data, batch_size=batch_size) # dataloader for the testing dataset

# Create the accuracy metric for estimating the goodness of the model.
metric = torchmetrics.Accuracy(task='multiclass', num_classes=10)

# Define the optimizer for gradient descent.
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # perform stochastic gradient descent on the parameters

# Define the training loop for the training stage.
def training_loop(dataloader, model, loss_function, optimizer):
    dataset_size = len(dataloader)
    # Start by loading the data batch from the disk.
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X) # get the predictions of the model
        # Compute the prediction error using the loss function.
        loss = loss_function(pred, y)
        # Perform backpropagation in order to minimize future losses.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss:{loss}, [{current} /{dataset_size}]")
            acc = metric(pred, y)
            print(f"Accuracy of the current batch: {acc}")
    # Print the final training accuracy.
    acc = metric.compute()
    print(f"Final training accuracy: {acc}")
    metric.reset() # reset for future loops

# Define the testing loop for the testing stage.
def testing_loop(dataloader, loss_function, model):
    # Remember to disable the weights update.
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            acc = metric(pred, y)
    acc = metric.compute()
    print(f"Final testing accuracy: {acc}")
    metric.reset()

# Perform the actual model training and testing.
for e in range(epochs):
    print(f"Epoch: {e}")
    training_loop(train_dataloader, model, loss_function, optimizer)
    testing_loop(test_dataloader, loss_function, model)
print("Done")

An autoencoder is a neural network model that is composed by an encoder, which compresses the input data to a lower-dimensional embedded feature, and a decoder, which uses the embedded feature in order to reconstruct the original data.

In [ ]:
import torch
from torch.utils.data import Dataloader
from torchvision.transforms import ToTensor
from torch import nn
from torchvision import datasets
import matplotlib.pyplot as plt
import torchmetrics

device = ('cuda' if torch.cuda.is_available() else 'cpu') # choose the device depending on GPU compatibility

# Start by importing and downloading the training and testing datasets.
training_data = datasets.MNIST(root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.MNIST(root='data', train=False, download=True, transform=ToTensor())

# Define the structure of the autoencoder through neural network classes.
# In particular, start by creating a sequential encoder and then "flip" its structure for the decoder.
class AutoEnc(nn.Module):
    def __init__(self):
        super.__init__()
        self.encoder = nn.Sequential(
            # nn.Flatten(),
            nn.Linear(28*28, 50)
            nn.ReLU(),
            nn.Linear(50, 25)
            nn.ReLU(),
            nn.Linear(25, 10)
        )
        self.decoder = nn.Sequential(
            nn.Linear(10, 25),
            nn.ReLU(),
            nn.Linear(25, 50),
            nn.ReLU(),
            nn.Linear(50, 28*28),
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# Create an instance of the model and define its hyperparameters.
model = AutoEnc()
epochs = 3
batch_size = 16
learning_rate = 0.001

# Then, define the loss function (and its optimizer) to use for the training stage.
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), learning_rate)

# Lastly, define the dataloader.
train_dataloader = Dataloader(training_data, batch_size=batch_size)
test_dataloader = Dataloader(test_data, batch_size=batch_size)

# At this point, start by defining the training loop for the training stage.
def training_loop(dataloader, model, loss_fn, optimizer):
    # Start by getting the batch of data to analyze.
    for batch, (X, y) in enumerate(dataloader):
        X = X.view(28, 28)
        X_reconstructed = model(X) # use the model to derive the reconstructed image
        loss = loss_fn(X_reconstructed, X) # compute the reconstruction error
        loss.backward() # perform the backward pass to upgrade the weights of the model
        optimizer.step()
        optimizer.zero_grad()
        if batch % 500 == 0:
            print(f'Current loss: {loss.item()}')

# Similarly, define the testing loop for the testing stage.
def test_loop(dataloader, model):
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X) # get the reconstructed image
            img = pred.view(-1, 28, 28)
            img = img.unsqueeze(1)
            img = img.permute(2, 3, 1, 0).to('cpu') # perform a procedure to print the reconstructed image
            plt.imshow(img[:,:,:,0]) # show the first image
            plt.show()

# Perform the actual training procedure.
for e in range(epochs):
    print(f'Epoch: {e}')
    training_loop(training_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model)